Q1.

(a) None

(b) '2026-05-06'

(c) ['2026-05-06', '2026-05-18']

(d) [('2026', '05', 06'), ('2026', '05', '18')]

(e) ['2026-05-06', '2026-05-18']

Q2.
(a) "[T]!"

(b) "[T]안녕[T] [T]세상[T]!"

(c) "[T]안녕[T] [T]세상[T]!"

(d) "수강생 <30>명, 조교 <3>명"

(e) "수강생 <\x01>명, 조교<\x01>명"

(ⅰ) (a)는 탐욕적 수량자를 써서 맨 처음 '<'부터 맨 끝 '>'을 잡아서 그 안을 모두 치환해버리고,
(b)는 게으른 수량자를 써서 최소 단위로 잡고 치환해버리기 때문이다.

(ⅱ) (e)는 (d)와 달리 repl부분이 원시 문자열이 아니어서 (d)는 '\1'을 첫번째 그룹으로 해석하지만 (e)는 8진수 이스케이프로 해석하기 때문이다.

In [ ]:
#Q3(1)
import re
import logging
from collections import Counter
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
#(a)
url = re.compile(r"https?://\S+")
html = re.compile(r"<[^>]+>")
email = re.compile(r"[\w.+-]+@[\w.-]+")
phone = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
mention = re.compile(r"@\w+")
hashtag = re.compile(r"#\w+")
k_alphabet = re.compile(r"[\u3131-\u3163]+")
space = re.compile(r"\s+")
#(a)
def clean_post(post: str) -> str:
    post = url.sub(" ", post)
    post = html.sub("", post)
    post = email.sub("[이메일]", post)
    post = phone.sub("[전화]", post)
    post = mention.sub(" ", post)
    post = hashtag.sub(" ", post)
    post = k_alphabet.sub("", post)
    post = space.sub(" ", post)
    return post.strip()
    
#(b)
def extract_hashtags(post: str) -> list[str]:
    return re.findall(r"#(\w+)", post)
#(c)
def analyze_posts(posts: list[str]) -> dict:
    i = [clean_post(post) for post in posts]
    posts_n: int = len(posts)
    try:
        avg_length_after_clean: float = round(sum(len(j) for j in i) / posts_n, 2)
        hashtag_list = []
        for post in posts:
            hashtag_list.extend(extract_hashtags(post))
        counter = Counter(hashtag_list)
        hashtag_counts: dict[str, int] = dict(counter.most_common())
        masked_count :int = 0
        for post in posts:
            post, email_count = email.subn("[이메일]", post)
            post, phone_count = phone.subn("[전화]", post)
            masked_count += (email_count + phone_count)
    except ZeroDivisionError:
        avg_length_after_clean = 0.0
        hashtag_counts = None
        masked_count = 0
        logging.warning(f"게시물이 없어서 '정제한 게시물 평균 길이', '모든 게시물의 해시태그 빈도', '모든 게시물에서 마스킹된 개인정보의 총 건수'를 각각 0.0, None, 0으로 처리함.")
    return {
        "게시물 개수": posts_n,
        "정제한 게시물 평균 길이": avg_length_after_clean,
        "모든 게시물의 해시태그 빈도": hashtag_counts,
        "모든 게시물에서 마스킹된 개인정보의 총 건수": masked_count
    }


In [42]:
#Q.3 (2)   
posts: list[str] = [
"오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ"
"자료: https://etl.snu.ac.kr/lec17",
"@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
"<b>중요</b>: 다음 시험 범위는 1-15강. "
"문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
" 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
"ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]
print([clean_post(post) for post in posts])
print(analyze_posts(posts))

['오늘 수업 진짜 재밌었음!! 감사 자료:', '팀플 어디서 모이지 카톡', '중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!', '여러 공백과 줄바꿈이 많은 텍스트', '진짜 좋다']
{'게시물 개수': 5, '정제한 게시물 평균 길이': 19.4, '모든 게시물의 해시태그 빈도': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, '모든 게시물에서 마스킹된 개인정보의 총 건수': 2}


Q3.(3)
re.compile을 사용해 많이 쓰이는 패턴들을 모았고, 6단계 순서대로 정제를 하도록 clean_post를 정의하였다. 그리고 extract_hashtags을 정의할 때는 '#'을 포함하지 않아야 하기에 hashtag을 그대로 쓰지 않고 r"#(\w+)"을 쓰고 re.findall함수를 사용하였다. 또 만약 posts가 아무것도 없는 list라면 posts_n이 0이므로 avg_length_after_clean에서 ZeroDivisionError가 나오기 때문에 굳이 posts_n이 0일때의 경우를 나누는 것보다 warning으로 알려주는 것이 좋을 것 같아 except구문을 사용하였다. 그리고 hashtag_counts에서는 counter 함수를 가져와서 내림차순으로 쉽게 정렬하도록 하였다. masked_count에서 re.subn함수를 사용하여 치환하면서 개수를 셀 수 있게 하였다. 6단계 정제 순서를 지켜야 하는 이유는 각 단계가 이전 단계의 결과에 영향을 받기 때문에 이 순서면 의도하지 않은 오류가 나오지 않는다. 예를 들어, 만약 단계 3보다 단계 4를 먼저 실행하게 되면, 이메일 주소의 @snu 부분이 멘션 패턴으로 해석해서 먼저 공백으로 잘려 나가게 되고, 그러면 이메일을 인식할 수 없게 된다.